# 04 Build Patient Features

This notebook builds AI-ready patient feature tables from normalized vital signs. It keeps feature engineering separate from the normalized relational import layer.

In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_DATA_DIR = Path('data/processed')
FEATURES_DATA_DIR = Path('data/features')

PATIENT_FILE = PROCESSED_DATA_DIR / 'patient.csv'
VITAL_SIGNS_FILE = PROCESSED_DATA_DIR / 'vital_signs.csv'

FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
patient_df = pd.read_csv(PATIENT_FILE)
vital_signs_df = pd.read_csv(VITAL_SIGNS_FILE, parse_dates=['measured_at'])

print('Patient shape:', patient_df.shape)
print('Vital signs shape:', vital_signs_df.shape)

In [ ]:
latest_vitals = (
    vital_signs_df
    .sort_values(['patient_id', 'vital_type', 'measured_at'])
    .groupby(['patient_id', 'vital_type'], as_index=False)
    .tail(1)
)

latest_vitals.head()

In [ ]:
latest_vitals_pivot = latest_vitals.pivot_table(
    index='patient_id',
    columns='vital_type',
    values='value',
    aggfunc='first'
).reset_index()

latest_vitals_pivot.columns.name = None
latest_vitals_pivot.head()

In [ ]:
recent_30d = vital_signs_df.copy()
max_ts = recent_30d['measured_at'].max()
cutoff_ts = max_ts - pd.Timedelta(days=30)
recent_30d = recent_30d[recent_30d['measured_at'] >= cutoff_ts].copy()

mean_30d = recent_30d.pivot_table(
    index='patient_id',
    columns='vital_type',
    values='value',
    aggfunc='mean'
).reset_index()

mean_30d = mean_30d.add_prefix('avg_30d_')
mean_30d = mean_30d.rename(columns={'avg_30d_patient_id': 'patient_id'})
mean_30d.head()

In [ ]:
weight_rows = vital_signs_df[vital_signs_df['vital_type'] == 'WEIGHT'].copy()
weight_rows = weight_rows.sort_values(['patient_id', 'measured_at'])

weight_trend = weight_rows.groupby('patient_id').agg(
    first_weight=('value', 'first'),
    latest_weight=('value', 'last'),
    first_weight_time=('measured_at', 'first'),
    latest_weight_time=('measured_at', 'last')
).reset_index()

weight_trend['weight_change'] = weight_trend['latest_weight'] - weight_trend['first_weight']
weight_trend.head()

In [ ]:
patient_features_df = patient_df[['id', 'patient_number', 'birth_date', 'gender']].copy()
patient_features_df = patient_features_df.rename(columns={'id': 'patient_id'})

patient_features_df = patient_features_df.merge(latest_vitals_pivot, on='patient_id', how='left')
patient_features_df = patient_features_df.merge(mean_30d, on='patient_id', how='left')
patient_features_df = patient_features_df.merge(weight_trend[['patient_id', 'weight_change', 'latest_weight_time']], on='patient_id', how='left')

patient_features_df.head()

In [ ]:
feature_output_file = FEATURES_DATA_DIR / 'patient_features.csv'
patient_features_df.to_csv(feature_output_file, index=False)

print('Exported:', feature_output_file)

In [ ]:
display(patient_features_df.head(20))